# Session 7 · Optional workbook: honest validation of the churn model

*Author: course team · Licence: CC-BY-4.0*

The core practice of Session 7 is the Berlin price model in [20-case-study-airbnb-price-validation.ipynb](20-case-study-airbnb-price-validation.ipynb). This optional workbook applies the same tools to the logistic churn model of Session 6, which Sessions 8 and 9 develop further.

| Part | Block | Task |
|---|---|---|
| A | 0:00–0:45 | Cross-validate the churn model and report ROC AUC with a bootstrap interval |
| B | 1:00–1:45 | Tune the regularisation `C` (and `class_weight`) with a grid search and judge how flat the optimum is |
| C | 2:00–2:45 | Estimate the tuned model without optimism: nested cross-validation |

**Data.** The IBM Telco churn data (7,043 customers), downloaded from IBM's GitHub repository. The notebook runs in well under a minute.

Tasks for you are marked **Task**.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

A helper for the bootstrap confidence interval of any metric on a fixed set of predictions (theory page 01).

In [ ]:
def bootstrap_ci(y_true, y_pred, metric, n_boot=1000, seed=0):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        stats.append(metric(y_true[i], y_pred[i]))
    return np.percentile(stats, [2.5, 97.5])

## Part A · Cross-validated scores with a confidence interval

We report cross-validated ROC AUC and a bootstrap interval on a stratified 25 % test set.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y_churn = (telco["Churn"] == "Yes").astype(int)
X_churn = telco.drop(columns=["customerID", "Churn"])
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in X_churn.columns if c not in num]
churn_model = make_pipeline(
    make_column_transformer((StandardScaler(), num), (OneHotEncoder(handle_unknown="ignore"), cat)),
    LogisticRegression(max_iter=1000))

skf = StratifiedKFold(5, shuffle=True, random_state=0)
auc = cross_val_score(churn_model, X_churn, y_churn, cv=skf, scoring="roc_auc")
print("AUC per fold:", auc.round(3), " mean", auc.mean().round(3), " sd", auc.std().round(3))

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_churn, y_churn, test_size=0.25, stratify=y_churn, random_state=0)
proba = churn_model.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]
print("test AUC:", round(roc_auc_score(yc_te, proba), 3),
      " 95% CI:", bootstrap_ci(yc_te, proba, roc_auc_score).round(3))

**Task A.** Collect the results in one table (metric, CV mean ± sd, test score, 95 % interval). Is the test score inside the range you would expect from cross-validation? Which of the two summaries describes the uncertainty of the test score, and which the variation between training sets?

In [ ]:
# your table here, for example:
# pd.DataFrame([{"model": "logistic regression (churn)", "metric": "ROC AUC", "cv_mean": ..., "cv_sd": ..., "test": ..., "ci_low": ..., "ci_high": ...}])

## Part B · Tuning `C`

Logistic regression in scikit-learn is regularised; `C = 1/α`, so a *small* `C` means a *strong* penalty. We search `C` on a log grid together with `class_weight`, and look at how flat the optimum is.

In [ ]:
grid = GridSearchCV(churn_model, {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10],
                                  "logisticregression__class_weight": [None, "balanced"]},
                    cv=skf, scoring="roc_auc")
grid.fit(Xc_tr, yc_tr)
res = pd.DataFrame(grid.cv_results_)[["param_logisticregression__C", "param_logisticregression__class_weight",
                                      "mean_test_score", "std_test_score"]]
print(res.sort_values("mean_test_score", ascending=False).round(4).to_string(index=False))
print("best:", grid.best_params_, " test AUC:", round(roc_auc_score(yc_te, grid.predict_proba(Xc_te)[:, 1]), 3))

**Task B.** Which `C` was chosen? Is the optimum flat or sharp compared with the standard deviation across folds? If the best value lies at the edge of the grid, extend the grid and search again. Why does `class_weight` hardly change ROC AUC (hint: AUC does not depend on the threshold; Session 8)?

## Part C · Nested cross-validation

`grid.best_score_` is the score of the setting that won on these very folds, so it is slightly optimistic. Nested cross-validation (theory page 03) puts the whole search inside an outer loop.

In [ ]:
inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
tuned = GridSearchCV(churn_model, {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]}, cv=inner, scoring="roc_auc")
nested = cross_val_score(tuned, X_churn, y_churn, cv=outer, scoring="roc_auc")
print("nested AUC per outer fold:", nested.round(3), " mean", nested.mean().round(4))
tuned.fit(X_churn, y_churn)
print("non-nested best_score_:", round(tuned.best_score_, 4), tuned.best_params_)

**Task C.** Compare `best_score_` with the nested mean. In our run the two differed by about 0.002 AUC, and `best_score_` was even slightly *lower* (it uses 3 folds, the nested estimate 5 outer folds): the optimism of choosing among five settings is far smaller than the spread across folds. In which kind of project would the optimism be large?

**Team project.** If your project predicts churn or another yes/no outcome, write down in your validation plan which splitter, which metric and which interval you will report.